In [ ]:
"""
Smoke test for shortest path calculation between concept pairs.
Shows examples with path 0, 1, 2, 3, and >3 if available.
"""

import sys
from pathlib import Path
import importlib

# Add parent directory to path for imports (tests/ is one level deep, need to go up 2)
sys.path.insert(0, str(Path.cwd().parent))

from scripts.process_elsst_train_test import load_concepts_optimized
from scripts import config
from collections import deque

# Reload config to get latest changes
importlib.reload(config)

# Use resolved path from config
RDF_FILE_PATH = str(config.PROJECT_ROOT / "datasets/raw_datasets/ELSST_R5.rdf")

# Load concepts
concepts_list, adj = load_concepts_optimized(RDF_FILE_PATH, lang="en")

# Create a mapping from ID to ConceptData for easier access
concepts = {i: c for i, c in enumerate(concepts_list)}

def find_shortest_path_bfs(id1, id2, adj):
    """BFS to find shortest path length between two concept IDs."""
    if id1 == id2:
        return 0
    visited = {id1}
    queue = deque([(id1, 0)])
    while queue:
        current, dist = queue.popleft()
        neighbors = adj.get(current, set())
        for neighbor in neighbors:
            if neighbor == id2:
                return dist + 1
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))
    return float('inf')  # No path found


def get_path_sequence_bfs(id1, id2, adj):
    """BFS to reconstruct the full path between two concept IDs."""
    if id1 == id2:
        return [id1]
    visited = {id1}
    queue = deque([(id1, [id1])])
    while queue:
        current, path = queue.popleft()
        neighbors = adj.get(current, set())
        for neighbor in neighbors:
            if neighbor == id2:
                return path + [neighbor]
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    return []


results = {0: [], 1: [], 2: [], 3: [], 'gt3': []}

# Try to find pairs for each path length
n_concepts = len(concepts_list)
for i in range(min(50, n_concepts)):
    for j in range(i+1, min(i+50, n_concepts)):
        if i == j:
            continue
        path = find_shortest_path_bfs(i, j, adj)
        if path == 0 and len(results[0]) < 2:
            results[0].append((i, j, path))
        elif path == 1 and len(results[1]) < 2:
            results[1].append((i, j, path))
        elif path == 2 and len(results[2]) < 2:
            results[2].append((i, j, path))
        elif path == 3 and len(results[3]) < 2:
            results[3].append((i, j, path))
        elif path > 3 and path != float('inf') and len(results['gt3']) < 2:
            results['gt3'].append((i, j, path))
        if all(len(results[k]) >= 2 for k in results):
            break
    if all(len(results[k]) >= 2 for k in results):
        break


print("Smoke test: Shortest path between concept pairs (with full path)")
for k in [0, 1, 2, 3, 'gt3']:
    for id1, id2, pathlen in results[k]:
        label1 = concepts_list[id1].pref if id1 < len(concepts_list) else "Unknown"
        label2 = concepts_list[id2].pref if id2 < len(concepts_list) else "Unknown"
        path_ids = get_path_sequence_bfs(id1, id2, adj)
        path_labels = [concepts_list[uid].pref if uid < len(concepts_list) else f"ID:{uid}" for uid in path_ids]
        print(f"Pair: '{label1}' <-> '{label2}' | Shortest path: {pathlen}")
        print("  Path: " + " -> ".join(path_labels))


# Specific test for LOANS <-> ADMINISTRATION OF JUSTICE
def find_id_by_label(label):
    for idx, c in enumerate(concepts_list):
        if c.pref == label:
            return idx
    return None


id_loans = find_id_by_label('LOANS')
id_justice = find_id_by_label('ADMINISTRATION OF JUSTICE')
if id_loans is not None and id_justice is not None:
    pathlen = find_shortest_path_bfs(id_loans, id_justice, adj)
    path_ids = get_path_sequence_bfs(id_loans, id_justice, adj)
    path_labels = [concepts_list[uid].pref if uid < len(concepts_list) else f"ID:{uid}" for uid in path_ids]
    print(f"\nPair: 'LOANS' <-> 'ADMINISTRATION OF JUSTICE' | Shortest path: {pathlen}")
    print("  Path: " + " -> ".join(path_labels))
else:
    print("\nCould not find both 'LOANS' and 'ADMINISTRATION OF JUSTICE' in the concepts.")

[1/8] Loading RDF into memory...
  Indexing Concept URIs...
  Extracting labels and relations (Vectorized)...
  Deriving narrower relations...
Smoke test: Shortest path between concept pairs (with full path)
Pair: 'LANGUAGES AND LINGUISTICS EDUCATION' <-> 'FOREIGN LANGUAGES AND CULTURES EDUCATION' | Shortest path: 1
  Path: LANGUAGES AND LINGUISTICS EDUCATION -> FOREIGN LANGUAGES AND CULTURES EDUCATION
Pair: 'HEROIN' <-> 'TRANQUILLIZERS' | Shortest path: 2
  Path: HEROIN -> DRUG USE -> TRANQUILLIZERS
Pair: 'COMBATIVE SPORTS' <-> 'RACKET GAMES' | Shortest path: 2
  Path: COMBATIVE SPORTS -> SPORTS -> RACKET GAMES
Pair: 'LISTENING' <-> 'TELEPHONE CALLS' | Shortest path: 3
  Path: LISTENING -> COMMUNICATION PROCESS -> INTERPERSONAL COMMUNICATION -> TELEPHONE CALLS
Pair: 'FOOTBALL TEAM SUPPORTERS' <-> 'PATIENTS' | Shortest path: 3
  Path: FOOTBALL TEAM SUPPORTERS -> SUBCULTURAL GROUPS -> GROUPS -> PATIENTS
Pair: 'TEACHING PROFESSION' <-> 'LOANS' | Shortest path: 9
  Path: TEACHING PROFESSI